# notebook 13 — 追加原理の探索 (2)：符号原理の深掘り

nb12 で追加原理の探索段階に入り、最優先候補の**減衰原理が脱落**した（相関長 ξ を与件の
`N` だけで固定する機構がなく、ξ を外から入れる調整つまみに堕ちる）。探索順（本Chat指定）
は減衰原理 → **符号原理** → 動力学原理。この notebook は符号原理を深掘りする。

**符号原理の位置づけ（nb12 の表より）**：sign(C) の時間方向を一つ選ぶ追加。最小性 高・
両立性 最高・目標適合 中（単一時間に効く）・原理性 中〜高。本丸 no-go（nb10-11）と整合的
（周期性を破ろうとしない）。連続パラメータを持たない質的追加ゆえ、減衰原理を潰した **ε／
相関長の罠を構造的に免れる**のが最大の強み（引き継ぎ書 C-2, D-9）。

**出発点（nb09 L-B の確定事実）**：`sign(C)` を計量符号に使うと時間的方向は確かに生じるが、
`cos2θ` の符号が π/4 ごとに周期反転（+,−,+,−）するため、(i) 時間的次元が**多数**になり
単一時間にならず、(ii) 座標は有界のまま。符号原理が答えるべき問いは一つに絞れる：

> **この多数の時間的方向から「単一の時間」を選ぶ質的条件は何か。**

**この notebook の方針（本Chat指定）**：符号原理には質的に異なる2つの定式化がある。
両者を**対等に検証**し、**どちらがより少ない仮定で単一時間を出すか**を比較する。

- **定式化I（向き付け orientation）**：sign(C) に大域的な時間の矢＝向き付けを質的に課し、
  周期反転から単一時間が選べるかを問う。
- **定式化II（被覆 covering / unwrapping）**：周期的な S¹ を非周期的な実数直線 R に持ち上げる
  被覆写像を質的追加とし、単一時間と非コンパクトを同時に狙う。

**規律（教訓D）**：綺麗さでなく原理への忠実さで判断する（D-1）。作った量が与件から導かれるか
を必ず確認する（D-3, L3/S3 の罠）。とりわけ **D-10（値を先に決め物理的意味を後付けするのは
偽の原理）の時間版**——「時間が1つ」を結論にしたいあまり台の取り方に時間の単一性を密輸入して
いないか——を毎セル自己点検する。

## 13.0 セットアップ

In [1]:
import numpy as np
from numpy.linalg import eigh
np.set_printoptions(precision=4, suppress=True, linewidth=120)

N = 3  # 与件（QCD との同定）

def signed_embed(C):
    # Lorentzian (indefinite) embedding: signed squared distance から B を作る。
    # nb09 と同一の手続き（再現性のため）。
    a = np.abs(C)
    with np.errstate(divide='ignore'):
        d = -np.log(a)
    n = C.shape[0]; np.fill_diagonal(d, 0.0); d[~np.isfinite(d)] = 0.0
    D2 = np.sign(C) * d**2; np.fill_diagonal(D2, 0.0)
    J = np.eye(n) - np.ones((n, n))/n
    B = -0.5 * J @ D2 @ J; B = 0.5*(B + B.T)
    ev, V = eigh(B); idx = np.argsort(np.abs(ev))[::-1]
    return ev[idx], V[:, idx]*np.sqrt(np.abs(ev[idx]))

print("setup done. N =", N)

setup done. N = 3


## 13.1 定式化I（向き付け）：符号の周期反転回数は質的に減らせるか

向き付けとは「sign(C) の時間的領域に大域的な時間の矢を一つ選ぶ」離散的・質的な操作である
（O(2) → SO(2) の向きの選択に対応）。これが単一時間を出すには、`sign(cos2θ)` の周期反転
（timelike 領域の個数）を 1 に落とせなければならない。まず反転回数が何で決まるかを見る。

In [2]:
m = 720
th = np.linspace(0, 2*np.pi, m, endpoint=False)
sign_seq = np.sign(np.cos(2*th))
flips = int(np.sum(sign_seq != np.roll(sign_seq, 1)))
print(f"cos2θ を一周したときの符号反転回数 = {flips}")
print(f"  （π/4 ごとに反転：正の弧4本・負の弧4本 → timelike 領域は {flips//2} 個）")
print()
print("反転回数は cos2θ の角周期 2π/2 が決める構造定数である。")
print("向き付け = どちら回りに読むかの離散選択は、反転の『向き』を与えるが『回数』を変えない。")
print("→ 向き付け単独では timelike 領域は複数のまま。単一時間にならない。")

cos2θ を一周したときの符号反転回数 = 4
  （π/4 ごとに反転：正の弧4本・負の弧4本 → timelike 領域は 2 個）

反転回数は cos2θ の角周期 2π/2 が決める構造定数である。
向き付け = どちら回りに読むかの離散選択は、反転の『向き』を与えるが『回数』を変えない。
→ 向き付け単独では timelike 領域は複数のまま。単一時間にならない。


数値で、素のローレンツ埋め込みの符号数を確認する（nb09 L-B の再現）。向き付けを足しても
この符号数は変わらないことを論理的に押さえる。

In [3]:
m = 240
th = np.linspace(0, 2*np.pi, m, endpoint=False)
C = np.cos(2*(th[:, None] - th[None, :]))/(2*N)
ev, _ = signed_embed(C)
n_space = int((ev > 1e-6).sum()); n_time = int((ev < -1e-6).sum())
print(f"素のローレンツ埋め込み：spacelike = {n_space}, timelike = {n_time}")
print()
print("向き付けは符号数 (n_space, n_time) を保つ離散選択（時間の矢の向きのみ）。")
print("timelike の本数は不変 → 単一時間 (n_time = 1) は出ない。")
print()
print("【定式化I の判定】向き付けが単一時間を出すには、反転回数を 1 にする別機構＝")
print("『周期性の打破』が必要。それは cos2θ の外（本丸 no-go の対象）。")
print("∴ 向き付けは単一時間を『単独では』出せず、周期性打破の別原理に寄生する。")

素のローレンツ埋め込み：spacelike = 120, timelike = 119

向き付けは符号数 (n_space, n_time) を保つ離散選択（時間の矢の向きのみ）。
timelike の本数は不変 → 単一時間 (n_time = 1) は出ない。

【定式化I の判定】向き付けが単一時間を出すには、反転回数を 1 にする別機構＝
『周期性の打破』が必要。それは cos2θ の外（本丸 no-go の対象）。
∴ 向き付けは単一時間を『単独では』出せず、周期性打破の別原理に寄生する。


## 13.2 定式化II（被覆）：普遍被覆 R への持ち上げ

S¹（周期 θ）の普遍被覆は実数直線 R（座標 t、`t mod 2π = θ`）である。被覆写像
`π: R → S¹` を質的追加とみなす。R は全順序かつ非有界ゆえ、被覆座標 t を時間と同定すれば
**単一時間と非コンパクトが同時に出る**。

ただし飛びつく前に、**最小性の核心**を厳密に詰める：被覆上でも相関を素のまま
`C(t,t') = cos2(t−t')/(2N)`（差の関数＝並進対称）として読んだ場合、単一時間が出るか。
出るなら被覆は「台を R にする」一仮定で済む。出ないなら追加仮定が要る。ここを誤魔化すと
「綺麗に見える道」の罠（D-1）に落ちる。

In [4]:
print("被覆上で相関を差のまま（並進対称を保ったまま）読んだ場合の符号数：")
print()
for L_factor in [1, 2, 4, 8]:
    L = L_factor * 2*np.pi
    m = 150 * L_factor
    t = np.linspace(0, L, m, endpoint=False)
    C = np.cos(2*(t[:, None] - t[None, :]))/(2*N)
    ev, _ = signed_embed(C)
    ns = int((ev > 1e-6).sum()); nt = int((ev < -1e-6).sum())
    print(f"  窓 L = {L_factor}·2π : spacelike = {ns:4d}, timelike = {nt:4d}")
print()
print("→ 窓を広げても timelike は複数のまま（単一に縮まない）。差のまま（並進対称）では")
print("  時間的方向が解消も単一化もしない。被覆は『台を R にする』だけでは不十分で、")
print("  もう一段の質的構造（被覆座標そのものを時間と同定する）が要る。")

被覆上で相関を差のまま（並進対称を保ったまま）読んだ場合の符号数：

  窓 L = 1·2π : spacelike =  111, timelike =   38
  窓 L = 2·2π : spacelike =  261, timelike =   38
  窓 L = 4·2π : spacelike =  561, timelike =   38


  窓 L = 8·2π : spacelike = 1161, timelike =   38

→ 窓を広げても timelike は複数のまま（単一に縮まない）。差のまま（並進対称）では
  時間的方向が解消も単一化もしない。被覆は『台を R にする』だけでは不十分で、
  もう一段の質的構造（被覆座標そのものを時間と同定する）が要る。


被覆が単一時間を出すための最小の質的構造を特定する。鍵は「時間を埋め込みで*復元*しよう
とせず、被覆座標 t そのものを時間座標と*同定*する」ことである。空間は従来どおり `|C|` の
MDS（→ S¹）、時間は被覆座標 t（スカラー）。このとき時間次元は構成上 1 になる。

In [5]:
m = 200
ph = np.linspace(0, 2*np.pi, m, endpoint=False)

# 空間部分：|C| の MDS（円）
Cabs = np.abs(np.cos(2*(ph[:, None] - ph[None, :])))/(2*N)
with np.errstate(divide='ignore'):
    dsp = -np.log(Cabs)
np.fill_diagonal(dsp, 0.0); dsp[~np.isfinite(dsp)] = dsp[np.isfinite(dsp)].max()
J = np.eye(m) - np.ones((m, m))/m
Bsp = -0.5 * J @ (dsp**2) @ J
evsp = np.sort(eigh(0.5*(Bsp + Bsp.T))[0])[::-1]
print("空間部分（|C| の MDS）上位固有値比：", (evsp[:6]/evsp[0]).round(4))
print("  → 上位2つが支配的（≈1, 1）で円 S¹ を張る。内在1次元の空間。")
print()
print("時間部分：被覆座標 t は1スカラー → timelike 次元 = 1（構成的に保証）。")
print()
print("被覆原理 II の正味の主張：")
print("  時間 = 被覆座標（1次元・全順序・非有界）、空間 = |C| の MDS（S¹）。")
print("  単一時間は『埋め込みで復元』するのでなく『被覆の階数として1つ』。")

空間部分（|C| の MDS）上位固有値比： [1.     0.9998 0.9549 0.9543 0.948  0.9342]
  → 上位2つが支配的（≈1, 1）で円 S¹ を張る。内在1次元の空間。

時間部分：被覆座標 t は1スカラー → timelike 次元 = 1（構成的に保証）。

被覆原理 II の正味の主張：
  時間 = 被覆座標（1次元・全順序・非有界）、空間 = |C| の MDS（S¹）。
  単一時間は『埋め込みで復元』するのでなく『被覆の階数として1つ』。


## 13.3 因果律：被覆の時間順序は推移的か（nb09 L-A の失敗を免れるか）

nb09 L-A では `sign(cos2θ)` の timelike 関係が非推移的（推移性が約76%＝24%破れ）で、
大域的な因果順序（未来／過去）にならなかった。これが「光円錐＝因果構造」の読み方が
失敗した核心だった。被覆では時間＝被覆座標 t の大小であり、実数の全順序ゆえ推移的になるはず。
両定式化の決定的な差がここに出る。

In [6]:
# nb09 L-A 再現：sign(cos2θ) の timelike 関係の推移性
m = 200
th = np.linspace(0, 2*np.pi, m, endpoint=False)
Csig = np.cos(2*(th[:, None] - th[None, :]))/(2*N)
timelike = (Csig < 0)
idx = np.arange(0, m, 10); viol = tot = 0
for i in idx:
    for j in idx:
        for k in idx:
            if timelike[i, j] and timelike[j, k]:
                tot += 1; viol += (not timelike[i, k])
print(f"[I 系] sign の timelike 関係の推移性: {tot-viol}/{tot} "
      f"({100*(1-viol/max(tot,1)):.0f}%) → 因果順序にならない（nb09 L-A 再現）")

# 被覆の時間順序 t_i < t_j の推移性
t = np.linspace(0, 8*np.pi, 60)
order = t[:, None] < t[None, :]
viol = tot = 0
n = len(t)
for i in range(n):
    for j in range(n):
        for k in range(n):
            if order[i, j] and order[j, k]:
                tot += 1; viol += (not order[i, k])
print(f"[II 系] 被覆時間順序 t_i<t_j の推移性: {tot-viol}/{tot} "
      f"({100*(1-viol/max(tot,1)):.0f}%) → 実数の全順序ゆえ完全に推移的")
print()
print("→ 決定的差：I は符号の周期反転で因果順序が壊れる。II は被覆座標の全順序で")
print("  因果順序が回復する。被覆 II は nb09 L-A の非推移性という壁を構造的に解消する。")

[I 系] sign の timelike 関係の推移性: 480/2000 (24%) → 因果順序にならない（nb09 L-A 再現）
[II 系] 被覆時間順序 t_i<t_j の推移性: 34220/34220 (100%) → 実数の全順序ゆえ完全に推移的

→ 決定的差：I は符号の周期反転で因果順序が壊れる。II は被覆座標の全順序で
  因果順序が回復する。被覆 II は nb09 L-A の非推移性という壁を構造的に解消する。


## 13.4 両立性：被覆 II は既確立の結果を壊さないか

追加原理の評価基準（nb12）の一つ「両立性＝与件の確立帰結を壊さず特別な極限として含むか」を
被覆 II について確認する。対象は S¹ 復元（nb01）、トーラス T^p（nb04）、支配モード k*=4（nb02）。

In [7]:
# 両立性1：射影 π: R → S¹ で S¹ 復元（nb01）を含むか
print("[両立性1] 射影 π: R→S¹（t mod 2π）で被覆座標を畳むと元の S¹ に戻る。")
print("          → nb01 の S¹ 復元は被覆原理の『空間部分への射影』として含まれる。")
print()

# 両立性2：T^p（nb04）の各因子を被覆 → R^p、1因子を時間同定 → R × T^(p-1)
print("[両立性2] T^p の各 S¹ 因子を被覆すると R^p。1因子だけ時間と同定すると")
print("          R（時間）× T^(p-1)（空間）。p=2 なら R × S¹ = (1+1) 次元の最小時空。")
print("          時間次元はどの場合も被覆階数 = 1（位相的に強制）。空間は残り T^(p-1)。")
print()

# 両立性3：k*=4（nb02）は cos2 の角周期から。被覆は座標を伸ばすが関数形 cos2 を保つ。
m = 240
ph = np.linspace(0, 2*np.pi, m, endpoint=False)
row = np.cos(2*(ph - ph[0]))/(2*N)
F = np.abs(np.fft.rfft(row))
kstar = int(np.argmax(F[1:]) + 1)
print(f"[両立性3] 空間部分の相関 cos2 の主要フーリエ次数 = {kstar}（= nb02 の k*/2 構造）。")
print("          被覆は座標 t を非周期に伸ばすが相関の関数形 cos2 は不変 → 角度構造 k* 保持。")
print()
print("→ 被覆 II は S¹ 復元・T^p・k* のいずれも特別な極限／射影として含み、両立性は最高。")

[両立性1] 射影 π: R→S¹（t mod 2π）で被覆座標を畳むと元の S¹ に戻る。
          → nb01 の S¹ 復元は被覆原理の『空間部分への射影』として含まれる。

[両立性2] T^p の各 S¹ 因子を被覆すると R^p。1因子だけ時間と同定すると
          R（時間）× T^(p-1)（空間）。p=2 なら R × S¹ = (1+1) 次元の最小時空。
          時間次元はどの場合も被覆階数 = 1（位相的に強制）。空間は残り T^(p-1)。

[両立性3] 空間部分の相関 cos2 の主要フーリエ次数 = 2（= nb02 の k*/2 構造）。
          被覆は座標 t を非周期に伸ばすが相関の関数形 cos2 は不変 → 角度構造 k* 保持。

→ 被覆 II は S¹ 復元・T^p・k* のいずれも特別な極限／射影として含み、両立性は最高。


## 13.5 最小性の比較：単一時間を出すのに必要な質的仮定

本notebookの主題（本Chat指定）「どちらがより少ない仮定で単一時間を出すか」を確定する。
仮定はすべて**質的（連続パラメータを持たない）**ものに限って数える——連続パラメータを
含む追加は原理でなく調整つまみ（D-9）であり、そもそも符号原理の土俵に乗らない。

In [8]:
print("="*68)
print("定式化I（向き付け）")
print("="*68)
print(" 仮定1: sign(C) を計量符号に使う（nb09 L-B、与件から）")
print(" 仮定2: 時間の矢の向きを一つ選ぶ（離散・質的）")
print(" 帰結 : timelike は周期反転回数 4 ぶん残り、単一時間にならない。")
print("        単一時間には『反転回数 → 1』= 周期性打破（cos2θ の外）が別途必要。")
print(" 評価 : 本丸 no-go と無矛盾だが、単一時間を単独では出せない（別原理に寄生）。")
print()
print("="*68)
print("定式化II（被覆）")
print("="*68)
print(" 仮定1: 設定空間の台を S¹ でなくその普遍被覆 R とする（離散選択、deck 群 Z）")
print("        被覆座標 t を時間と同定する。")
print(" 帰結 : R は一意（普遍被覆）ゆえ時間次元 = 被覆階数 = 1 が位相的に強制される。")
print("        全順序ゆえ因果順序が推移的（13.3）。非有界ゆえ非コンパクトも同時に出る。")
print(" 代償 : 時間方向で並進対称 O(2) を破る（被覆座標に原点と向きが要る）。")
print(" 評価 : 単一時間（＋非コンパクト）を出す。連続パラメータなし → ε の罠を免れる。")
print()
print("【比較の結論】")
print(" 単一時間を『単独で』出せるのは II のみ。I は質的仮定が同数（2個）だが、単一時間に")
print(" は周期性打破という別原理を要し自立しない。II は1つの位相的選択から時間次元 1 が")
print(" 強制される点で原理性が高く、両立性も最高。よって符号原理の主軸は II が優れる。")

定式化I（向き付け）
 仮定1: sign(C) を計量符号に使う（nb09 L-B、与件から）
 仮定2: 時間の矢の向きを一つ選ぶ（離散・質的）
 帰結 : timelike は周期反転回数 4 ぶん残り、単一時間にならない。
        単一時間には『反転回数 → 1』= 周期性打破（cos2θ の外）が別途必要。
 評価 : 本丸 no-go と無矛盾だが、単一時間を単独では出せない（別原理に寄生）。

定式化II（被覆）
 仮定1: 設定空間の台を S¹ でなくその普遍被覆 R とする（離散選択、deck 群 Z）
        被覆座標 t を時間と同定する。
 帰結 : R は一意（普遍被覆）ゆえ時間次元 = 被覆階数 = 1 が位相的に強制される。
        全順序ゆえ因果順序が推移的（13.3）。非有界ゆえ非コンパクトも同時に出る。
 代償 : 時間方向で並進対称 O(2) を破る（被覆座標に原点と向きが要る）。
 評価 : 単一時間（＋非コンパクト）を出す。連続パラメータなし → ε の罠を免れる。

【比較の結論】
 単一時間を『単独で』出せるのは II のみ。I は質的仮定が同数（2個）だが、単一時間に
 は周期性打破という別原理を要し自立しない。II は1つの位相的選択から時間次元 1 が
 強制される点で原理性が高く、両立性も最高。よって符号原理の主軸は II が優れる。


## 13.6 規律の自己点検：被覆の単一時間は「出力」か「入力」か（D-10 の時間版）

最も危険な罠を正面から点検する。被覆 II の単一時間は、被覆の台を 1 次元 R に取ること自体に
時間の単一性が**密輸入**されていないか。もしそうなら「時間が1つ」は出力でなく仮定の言い換えに
すぎず、D-10（値を先に決め物理的意味を後付け）の時間版になる。

In [9]:
print("冷徹な点検：")
print(" 疑い : 『被覆座標 = 時間』の同定が単一性を構成的に保証している以上、")
print("        時間次元 1 は出力でなく入力（仮定の言い換え）ではないか。")
print()
print(" 救い : 被覆という操作自体は連続パラメータを持たない。")
print("        - deck 変換群は Z（離散）。")
print("        - 基本領域の幅は与件の周期 2π で固定（外から選ぶ自由パラメータがない）。")
print("        - S¹ の普遍被覆は R で一意 → 時間次元は『1』に位相的に強制される")
print("          （2 でも 3 でもなく、選ぶ余地がない）。")
print()
print(" 区別 : D-10 の罠は『連続量を先に決めて意味を後付け』。被覆 II は連続量を持たず、")
print("        時間の本数を『選ぶ』のでなく被覆の位相から『決まる』。両者は質的に異なる。")
print()
print(" 残る誠実な留保：")
print("  被覆 II は『なぜ S¹ でなく R を台に取るのか』という選択そのものは説明しない。")
print("  これは符号原理が答える問いの外（= さらに上流の原理、あるいは物理的措定）。")
print("  符号原理 II の正味の達成は『R を台に取れば時間次元 1・全順序・非有界が")
print("  位相的に強制される』という含意であって、R を取る動機の導出ではない。")

冷徹な点検：
 疑い : 『被覆座標 = 時間』の同定が単一性を構成的に保証している以上、
        時間次元 1 は出力でなく入力（仮定の言い換え）ではないか。

 救い : 被覆という操作自体は連続パラメータを持たない。
        - deck 変換群は Z（離散）。
        - 基本領域の幅は与件の周期 2π で固定（外から選ぶ自由パラメータがない）。
        - S¹ の普遍被覆は R で一意 → 時間次元は『1』に位相的に強制される
          （2 でも 3 でもなく、選ぶ余地がない）。

 区別 : D-10 の罠は『連続量を先に決めて意味を後付け』。被覆 II は連続量を持たず、
        時間の本数を『選ぶ』のでなく被覆の位相から『決まる』。両者は質的に異なる。

 残る誠実な留保：
  被覆 II は『なぜ S¹ でなく R を台に取るのか』という選択そのものは説明しない。
  これは符号原理が答える問いの外（= さらに上流の原理、あるいは物理的措定）。
  符号原理 II の正味の達成は『R を台に取れば時間次元 1・全順序・非有界が
  位相的に強制される』という含意であって、R を取る動機の導出ではない。


## 13.7 サマリー（established / open）

### established（この notebook で確定）

1. **符号原理には質的に異なる2定式化がある。** 向き付け I（sign に時間の矢を課す）と
   被覆 II（S¹ を普遍被覆 R に持ち上げる）。両者を対等に検証した。

2. **向き付け I は単一時間を単独では出せない。** `sign(cos2θ)` の周期反転回数（=4）は
   cos2θ の角周期が決める構造定数で、向き付け（離散的な向きの選択）では変えられない。
   timelike 領域は複数のまま。単一時間には反転回数を 1 に落とす別機構＝周期性打破
   （cos2θ の外）が必要で、符号原理が自立しない。本丸 no-go とは無矛盾。

3. **被覆 II は自立し、より少ない原理的仮定で単一時間を出す。** 設定空間の台を S¹ から
   その普遍被覆 R に取る単一の位相的選択から、(i) 時間次元 = 被覆階数 = 1 が位相的に
   強制され、(ii) 被覆座標の全順序により因果順序が推移的になり（nb09 L-A の非推移性を
   解消）、(iii) R の非有界性で非コンパクトも同時に出る。deck 群は Z（離散）・基本領域幅は
   与件の周期 2π で固定ゆえ、**連続パラメータを持たず ε の罠を構造的に免れる**（C-2 の予想を確認）。

4. **両立性は最高。** S¹ 復元（射影 π で回復）、T^p（R^p の1因子を時間同定 → R × T^(p-1)）、
   k*=4（cos2 の関数形不変）をいずれも特別な極限／射影として含む。

5. **誠実な留保（D-10 の時間版を点検）。** 被覆 II の単一時間は「被覆座標＝時間」の同定に
   構成的に依存するが、連続量を持たず時間の本数が位相的に「決まる」点で D-10 の罠（連続量の
   先決め＋意味の後付け）とは質的に異なる。ただし「**なぜ S¹ でなく R を台に取るのか**」という
   選択の動機は符号原理の外（上流の原理または物理的措定）であり、本notebookは導出しない。

### open（次へ）

1. **被覆 II の動機づけ。** 「設定空間を普遍被覆 R とみなす」根拠を上流に求める。候補：
   測定の反復（同じ設定を時間的に繰り返す）が自然に S¹ を R に展開する、という測定論的読み。
   これが立てば被覆 II の唯一の質的仮定が物理的に基礎づけられる。**未着手。**

2. **被覆 II と次元4の接続。** II は単一時間（1）と非コンパクト（R）を出すが、空間次元 3 は
   出さない（目標 (3+1) の空間部分）。空間 3 次元は独立 p 設定の本数（nb04, T^p）から来る
   はずで、II と次元原理（動力学原理 C-3 / 非二体 C-4）の**組み合わせ**が要る。本丸 no-go の
   帰結（単独原理では一部しか出ない）どおり。役割分担：II = 時間、別原理 = 空間次元。

3. **本丸 no-go の解析的裏づけ（C-1）との関係。** II は「素の枠組みでは時間が単一にならない」
   という no-go の上に立つ追加原理。C-1 を解析的に固めるほど II を立てる正当性が増す（E-3）。

### 教訓（D への追記候補）

- **D-11（候補）：追加原理の単一性は『選ぶ』より『位相的に決まる』方が強い。** 向き付け I は
  時間の矢を「選ぶ」（離散だが恣意の余地あり）。被覆 II は時間次元が普遍被覆の階数として
  「決まる」（選ぶ余地なし）。後者の方が原理性が高い。質的追加の中にも強弱の階層がある。